In [ ]:
import os

"""
DIRECTORIES
"""
CATS_DIRECTORY = "PetImages/Cat"
DOGS_DIRECTORY = "PetImages/Dog"

TRAIN_DIRECTORY = "dataset/train"
TEST_DIRECTORY = "dataset/test"
TRAIN_CATS_DIRECTORY = "dataset/train/cats"
TEST_CATS_DIRECTORY = "dataset/test/cats"
TRAIN_DOGS_DIRECTORY = "dataset/train/dogs"
TEST_DOGS_DIRECTORY = "dataset/test/dogs"

JPG_EXTENSION_FILTER = ".jpg"

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

In [6]:
import os

def get_images_with_extension(class_dir: str, extension: str = JPG_EXTENSION_FILTER):
    img_list = [
    img for img in os.listdir(class_dir)
    if img.endswith(extension) and os.path.isfile(os.path.join(class_dir, img))
    ]
    
    return img_list



In [21]:
import os
from torchvision.io import decode_image
from torchvision.transforms import ConvertImageDtype
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class CatDogDataset(Dataset):
    def __init__(self, imgs_dir, transform=None, target_transform=None):
        self.imgs_dir = imgs_dir
        self.transform = transform
        self.target_transform = target_transform
        self.images = []
        self.labels = []

        try:
            for label, class_name in enumerate(['cats', 'dogs']):
                class_dir = os.path.join(imgs_dir, class_name)

                for i, img_name in enumerate(get_images_with_extension(class_dir)):
                    self.images.append(os.path.join(class_dir, img_name))
                    self.labels.append(label)
                    if i % 1000 == 0:
                        print(f"Собрано {i} изображений")
                    
        except Exception as e:
            print(f"Ошибка сканирования папок: {e}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image_path = self.images[index]
        image = Image.open(image_path).convert('RGB')
        label = self.labels[index]

        if self.transform:
            image = self.transform(image)
        
        if self.target_transform:
            label = self.target_transform(label)

        return image, label


In [1]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel=3, stride=1, padding=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel, stride, padding),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class CatDogCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()

        # Feature extractor
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 32),
            nn.MaxPool2d(2),

            ConvBlock(32, 64),
            ConvBlock(64, 64),  
            nn.MaxPool2d(2),

            ConvBlock(64, 128),
            nn.MaxPool2d(2)
        )

        # Global pooler
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


In [19]:
import os
os.chdir(r'D:\Vova\Programming\Python projects\dogs-and-cats-classification')

In [23]:
import os
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn

img_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
])

train_dataset = CatDogDataset(TRAIN_DIRECTORY, transform=img_transform)

Собрано 0 изображений
Собрано 1000 изображений
Собрано 2000 изображений
Собрано 3000 изображений
Собрано 4000 изображений
Собрано 5000 изображений
Собрано 6000 изображений
Собрано 7000 изображений
Собрано 8000 изображений
Собрано 9000 изображений
Собрано 0 изображений
Собрано 1000 изображений
Собрано 2000 изображений
Собрано 3000 изображений
Собрано 4000 изображений
Собрано 5000 изображений
Собрано 6000 изображений
Собрано 7000 изображений
Собрано 8000 изображений
Собрано 9000 изображений


In [24]:
import os
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Using GPU for more faster model training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CatDogCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

total_params = sum(p.numel() for p in model.parameters())
print(f"Параметров в модели: {total_params}")

Параметров в модели: 173602
